# FIREWATCH AI — Delivery Integrity and Data Contract

**Stage 02.** Verify extracted files against ZIP members, quantify observation-date spans and gaps, and inspect schema requirements. This notebook is read-only. Run top-to-bottom from the project root or let it discover the root.

In [ ]:
from pathlib import Path
import hashlib, zipfile
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "2025").is_dir() and (p / "2026").is_dir(): return p
    raise FileNotFoundError("Cannot locate project root with 2025/ and 2026/ folders")
ROOT=find_root()
def sha256_stream(stream):
    h=hashlib.sha256()
    for chunk in iter(lambda: stream.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()
def file_hash(path):
    with open(path,"rb") as f: return sha256_stream(f)
print("Project root:", ROOT)

## ZIP member integrity

For each archive member, compare its SHA-256 digest with the extracted file. The notebook does not extract or overwrite files.

In [ ]:
integrity=[]
for zpath in sorted(ROOT.glob("*/*.zip")):
    with zipfile.ZipFile(zpath) as zf:
        for info in zf.infolist():
            if info.is_dir(): continue
            disk=zpath.parent/zpath.stem/info.filename
            with zf.open(info) as stream: zipped_hash=sha256_stream(stream)
            disk_hash=file_hash(disk) if disk.is_file() else None
            integrity.append({"zip":str(zpath.relative_to(ROOT)),"member":info.filename,
                              "zip_bytes":info.file_size,"extracted_exists":disk.is_file(),
                              "exact_sha256_match":zipped_hash==disk_hash})
integrity_df=pd.DataFrame(integrity)
display(integrity_df)
print(f"Verified {len(integrity_df)} members; all exact matches: {integrity_df.exact_sha256_match.all()}")

## Schema and date coverage

These CSVs contain detections, not a continuous satellite coverage record. The number of distinct dates with rows and the gaps between such dates are data-presence diagnostics only; a date with no CSV rows is not evidence that no fire occurred.

In [ ]:
files=sorted(p for y in ("2025","2026") for p in (ROOT/y).rglob("*.csv"))
def product(p): return next((x for x in ("M-C61","SV-C2","J1V-C2","J2V-C2") if x in p.name),"unknown")
def stream(p): return "archive" if "fire_archive_" in p.name else "near_real_time" if "fire_nrt_" in p.name else "unknown"
coverage=[]; schemas={}
for p in files:
    header=tuple(pd.read_csv(p,nrows=0).columns); schemas.setdefault(header,[]).append(str(p.relative_to(ROOT)))
    dates=pd.to_datetime(pd.read_csv(p,usecols=["acq_date"])["acq_date"],errors="coerce").dropna().dt.normalize()
    uniq=sorted(dates.unique())
    gaps=[(pd.Timestamp(b)-pd.Timestamp(a)).days-1 for a,b in zip(uniq,uniq[1:]) if (pd.Timestamp(b)-pd.Timestamp(a)).days>1]
    coverage.append({"year":p.parent.parent.name,"product":product(p),"stream":stream(p),
                     "file":str(p.relative_to(ROOT)),"records":len(dates),"unique_observation_dates":len(uniq),
                     "first_date":uniq[0] if uniq else pd.NaT,"last_date":uniq[-1] if uniq else pd.NaT,
                     "gaps_between_dates":len(gaps),"missing_calendar_days_between_rows":sum(gaps)})
coverage_df=pd.DataFrame(coverage)
display(coverage_df)
print("Distinct schemas:",len(schemas))
for cols,paths in schemas.items(): print(f"{len(cols)} columns: {', '.join(cols)}; files={len(paths)}")

## Archive/NRT calendar overlap

This checks whether the delivered archive and NRT snapshots contain rows on the same calendar date. A zero overlap in this snapshot does not define what future downloads do.


In [ ]:
# Compare calendar dates with at least one row in archive and NRT per year/product.
sets={}
for item in files:
    if stream(item)=="unknown": continue
    ds=set(pd.to_datetime(pd.read_csv(item,usecols=["acq_date"])["acq_date"]).dt.strftime("%Y-%m-%d"))
    sets.setdefault((item.parent.parent.name,product(item)),{})[stream(item)]=ds
comparisons=[]
for (year,prod),streams in sorted(sets.items()):
    if "archive" in streams and "near_real_time" in streams:
        archive=streams["archive"]; nrt=streams["near_real_time"]
        comparisons.append({"year":year,"product":prod,"overlap_dates":len(archive & nrt),
                            "archive_last_date":max(archive),"nrt_first_date":min(nrt),
                            "calendar_gap_days":max(0,(pd.Timestamp(min(nrt))-pd.Timestamp(max(archive))).days-1)})
display(pd.DataFrame(comparisons))

## Coverage timeline visualization

In [ ]:
fig,ax=plt.subplots(figsize=(12,max(4,len(coverage_df)*0.45)))
p=coverage_df.reset_index(drop=True)
for i,r in p.iterrows():
    if pd.notna(r.first_date) and pd.notna(r.last_date):
        ax.barh(i,(r.last_date-r.first_date).days+1,left=r.first_date,
                color="#e87924" if r.stream=="archive" else "#50b8c3")
ax.set_yticks(range(len(p))); ax.set_yticklabels([f"{r.year} · {r.product} · {r.stream}" for r in p.itertuples()])
ax.set_title("Span of dates containing at least one delivered observation")
ax.set_xlabel("Date (span does not imply daily coverage)"); ax.grid(axis="x",alpha=.25)
plt.tight_layout(); plt.show()

## Proposed normalized observation contract

| Field | Contract |
|---|---|
| `observation_id` | Stable ID derived from source checksum and row identity; never based on rounded coordinates alone. |
| `source_file`, `source_sha256` | Preserve relative path and exact source checksum. |
| `product_code` | Keep FIRMS product identity (`M-C61`, `SV-C2`, `J1V-C2`, `J2V-C2`). |
| `stream_type` | Keep archive/NRT identity distinct; preserve source rows and define overlap/update policy. |
| `source_satellite`, `source_instrument`, `source_version` | Preserve original values, alongside any normalized mapping. |
| `latitude`, `longitude` | Numeric decimal-degree coordinates; invalid values are quarantined with reason. |
| `acq_date`, `acq_time`, `observation_datetime_utc` | Preserve original tokens; derive UTC timestamp only after confirming FIRMS time semantics. |
| `confidence_raw`, `confidence_family` | Preserve original value and type; do not pool MODIS numeric and VIIRS categorical classes as one score. |
| `brightness`, `bright_t31`, `frp`, `scan`, `track`, `daynight`, `type` | Preserve source values and missingness. `type` is nullable because it is absent from the delivered NRT schema. |
| `ingested_at`, `pipeline_version` | Add to derived data for reproducibility. |

Inferred event associations must live in separate links/clusters with distances, time deltas, thresholds, method/version, and evidence; they must not overwrite a source observation.

## Stage 02 interpretation

A matching ZIP proves the extracted bytes agree with that downloaded bundle; it does not validate scientific content or FIRMS product semantics. The next stage will implement a reproducible source-to-derived ingestion pipeline only after the timestamp and stream policies are documented. No observation should be counted twice because archive and NRT are both present, and missing detection dates must not be turned into claims of zero fire activity.